# Reproduction Walkthrough by Workflow for the paper "Adapting deep learning phase detectors for seismic array processing".

This notebook follows `docs/reproduce_main_results.md` and is organized by workflow:
single-station, ensemble, beam, and array detection.

Each workflow section includes:
- targeted OSF data helper commands
- targeted OSF model/prediction artifact helper commands
- windowed test-data commands
- continuous-data commands

## Setup
Make sure the environment in `environment.yml` is installed.


In [ ]:
from pathlib import Path
import subprocess

def find_repo(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "environment.yml").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory")

repo = find_repo(Path.cwd().resolve())

def run(cmd: str) -> int:
    print("+", cmd)
    result = subprocess.run(cmd, cwd=repo, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result.returncode

def show_cmds(title: str, cmds):
    print(f"\n{title}")
    for idx, cmd in enumerate(cmds, start=1):
        print(f"{idx:2d}. {cmd}")

def run_cmds(cmds):
    for cmd in cmds:
        rc = run(cmd)
        if rc != 0:
            raise RuntimeError(f"Command failed: {cmd}")

print("Repo:", repo)

## 1) Single-Station Detection Workflow

Define commands first. Remove "--dry-run" to actually download data. This will overwrite dummy data from GIT repo!

In [ ]:
single_data_cmds = [
    "bash scripts/prepare_osf_data.sh --dry-run --pattern '1stat_2022_*.hdf5'"]
single_artifact_cmds = [
    "bash scripts/prepare_osf_artifacts.sh --dry-run --pattern 'saved_model_1stat_*.zip' --pattern 'predictions_1stat_*.npz'"]
single_windowed_cmds = [
    "python train.py --config config_1stat.yaml",
    "python evaluate_on_testdata.py --config config_1stat.yaml",
]
single_cont_cmds = [
    "python predict_continuous.py -c config_1stat.yaml",
    "python evaluate_continuous.py --config config_1stat.yaml",
]

Retrieve data, models and test data predictions:

In [ ]:
run_cmds(single_data_cmds)
run_cmds(single_artifact_cmds)

Train and evaluate on windowed event data (year 2022 only):

In [ ]:
run_cmds(single_windowed_cmds)

Detect and evaluate on continuous data from ARCES:

In [ ]:
run_cmds(single_cont_cmds)

## 2) Ensemble Detection Workflow

See Section 4.1.1 (test data) and 4.2 (continuous data) in the paper.


Note: OSF does not include all ensemble prediction archives due to size limits.
If missing, generate with `predict_on_testdata.py` on the provided 2022 data.

Define commands first. Remove "--dry-run" to actually download data. This will overwrite dummy data from GIT repo!

In [ ]:
ensemble_data_cmds = [
    "bash scripts/prepare_osf_data.sh --dry-run --pattern '1statfullarray_2022_*.hdf5'"
]
ensemble_artifact_cmds = [
    "bash scripts/prepare_osf_artifacts.sh --dry-run --pattern 'saved_model_1stat_*.zip'"
]
ensemble_windowed_cmds = [
    "python predict_on_testdata.py --config config_1statfullarray.yaml",
    "python evaluate_on_testdata.py --config config_1statfullarray.yaml",
]
ensemble_cont_cmds = [
    "python predict_continuous.py -c config_1statfullarray_cont.yaml",
    "python evaluate_continuous.py --config config_1statfullarray_cont.yaml",
]


Retrieve data, models and test data predictions:


In [ ]:
run_cmds(ensemble_data_cmds)
run_cmds(ensemble_artifact_cmds)


Predict wth single station model on all array stations, combine output, and evaluate on windowed event data (year 2022 only):


In [ ]:
run_cmds(ensemble_windowed_cmds)


Detect and evaluate on continuous data from ARCES:


In [ ]:
run_cmds(ensemble_cont_cmds)


## 3) Beam Detection Workflow

See Section 4.1.2 (test data) and 4.2 (continuous data) in the paper.

Define commands first. Remove "--dry-run" to actually download data. This will overwrite dummy data from GIT repo!


In [ ]:
beam_data_cmds = [
    "bash scripts/prepare_osf_data.sh --dry-run --pattern 'zbeam_2022_*.hdf5' --pattern '3cbeam_2022_*.hdf5'"
]
beam_artifact_cmds = [
    "bash scripts/prepare_osf_artifacts.sh --dry-run --pattern 'saved_model_zbeam_*.zip' --pattern 'predictions_zbeam_*.npz' --pattern 'saved_model_3cbeam_*.zip' --pattern 'predictions_3cbeam_*.npz'"
]
beam_windowed_cmds = [
    "python train.py --config config_zbeam.yaml",
    "python evaluate_on_testdata.py --config config_zbeam.yaml",
    "python train.py --config config_3cbeam.yaml",
    "python evaluate_on_testdata.py --config config_3cbeam.yaml",
]
beam_cont_cmds = [
    "python predict_continuous.py -c config_zbeam.yaml",
    "python evaluate_continuous.py --config config_zbeam.yaml",
    "python predict_continuous.py -c config_3cbeam.yaml",
    "python evaluate_continuous.py --config config_3cbeam.yaml",
]


Retrieve data, models and test data predictions:


In [ ]:
run_cmds(beam_data_cmds)
run_cmds(beam_artifact_cmds)


Train and evaluate on windowed event data (year 2022 only):


In [ ]:
run_cmds(beam_windowed_cmds)


Detect and evaluate on continuous data from ARCES:


In [ ]:
run_cmds(beam_cont_cmds)


## 4) Array Detection Workflow (ARCES Set 2)

See Section 4.1.3 (test data) and 4.2 (continuous data) in the paper.

Define commands first. Remove "--dry-run" to actually download data. This will overwrite dummy data from GIT repo!


Note: OSF does not include all array prediction archives due to size limits.
If missing, generate with `predict_on_testdata.py` on the provided 2022 data.


In [ ]:
array_data_cmds = [
    "bash scripts/prepare_osf_data.sh --dry-run --pattern 'arces25_2022_*.hdf5' --pattern 'array25arces_2022_array_waveforms_000*.hdf5'"
]
array_artifact_cmds = [
    "bash scripts/prepare_osf_artifacts.sh --dry-run --pattern 'saved_model_arrayarces_set2_*.zip' --pattern 'predictions_arrayarces_set2_*.npz'"
]
array_windowed_cmds = [
    "python train.py --config config_arrayarces_set2.yaml",
    "python predict_on_testdata.py --config config_arrayarces_set2.yaml",
    "python evaluate_on_testdata.py --config config_arrayarces_set2.yaml",
]
array_cont_cmds = [
    "python predict_continuous.py -c config_arrayarces_set2.yaml",
    "python evaluate_continuous.py --config config_arrayarces_set2.yaml",
]


Retrieve data, models and test data predictions:


In [ ]:
run_cmds(array_data_cmds)
run_cmds(array_artifact_cmds)


Train, predict, and evaluate on windowed event data (year 2022 only):


In [ ]:
run_cmds(array_windowed_cmds)


Detect and evaluate on continuous data from ARCES:


In [ ]:
run_cmds(array_cont_cmds)


## Continuous quick-test vs full interval

Current configs are mostly set to a 1-hour quick-test interval.
For full 4-day labeled continuous evaluation, update `prediction.end_time` in
the config to `2023-01-05T00:00:00` (or your desired end time).